In [104]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import numpy as np
import subprocess
import sys

import os
import pickle

import seaborn as sns
from sklearn.metrics import precision_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.tree import DecisionTreeRegressor, plot_tree, _tree
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

import math
import scipy.cluster as sc
import scipy.spatial.distance as sd
from tqdm.auto import tqdm

In [ ]:
def installPackage(package):
    try:
        subprocess.check_call([sys.executable, '-m', 'conda', 'install', package, '-y'])
    except subprocess.CalledProcessError as e:
        print(f'Error installing {package}: {e}')

def checkAndinstallPackages():
    required = [
        'pandas', 'matplotlib','numpy',
        'seaborn', 'scikit-learn', 'scipy',
        'math', 'tqdm'
    ]

    for package in tqdm(required, desc='Checking and Installing Packages', unit='package'):
        try:
            __import__(package)
            print(f'{package} is already installed')
        except ImportError:
            print(f'{package} not found, installing')
            subprocess.check_call([sys.executable, "-m", "conda", "install", package, "-y"])

checkAndinstallPackages()

In [ ]:
filePath = 'Top10VideoGameStocks.csv'
vG = pd.read_csv(filePath, sep=',')
vG.head(5)

In [ ]:
vG= vG.rename(columns={'Ticker Symbol': 'Ticker_Symbol', 'Adj Close': 'Adj_Close'})
print(vG['Ticker_Symbol'].unique())

In [ ]:
companyTickers = ['SONY', '0700.HK', 'MSFT', 'NTDOY', 'NTES', 'EA', 'TTWO',
       'EMBRAC-B.ST', 'RBLX', 'PLTK']

def filterByCompany(filePath, TickerName):
    df = filePath
    if TickerName not in companyTickers:
        print(f"Company Ticker '{companyTickers}' is not in the list of valid Company Tickers.")
        return None
    filteredData = df[df['Ticker_Symbol'] == TickerName]
    dictName = TickerName.replace(" ", "_") + "_filtered"
    globals()[dictName] = filteredData

    return filteredData

for Ticker in companyTickers:
    filterByCompany(vG, Ticker)

filteredDatasets = [var for var in globals() if var.endswith('_filtered')]
print(filteredDatasets)

In [ ]:
SONY_filtered = SONY_filtered.set_index('Date')
SONY_filtered

In [ ]:
SONY_filtered.plot(y='Close', use_index='True')

In [9]:
SONY_filtered['Tomorrow'] = SONY_filtered['Close'].shift(-1)
SONY_filtered['Target'] = (SONY_filtered['Tomorrow'] > SONY_filtered['Close']).astype(int) # is tomorrows price greater than today

In [ ]:
model0 = RandomForestClassifier(n_estimators=250, min_samples_split=7, max_depth = 10, random_state=31)

train0= SONY_filtered.iloc[:-60]
test0= SONY_filtered.iloc[-238:]

predictors = ['Close', 'Volume', 'High', 'Low']
target = 'Target'
model.fit(train[predictors], train['Target'])

predictions = model.predict(test[predictors])
predictions = pd.Series(predictions, index=test.index)
accuracy = accuracy_score(test[target], predictions)
print(f'Accuracy: {accuracy}')

precision = precision_score(test['Target'], predictions)
print(f'Precision Score: {precision}')

tree = model.estimators_[0]

def getMaxDepth(tree):
    return tree.tree_.max_depth



In [ ]:
threshold = 0.5  
train0.loc[:, 'Target'] = (train0['Target'] > threshold).astype(int)
test0.loc[:, 'Target'] = (test0['Target'] > threshold).astype(int)

train_encoded = pd.get_dummies(train0.drop('Target', axis=1))
test_encoded = pd.get_dummies(test0.drop('Target', axis=1))

train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)
 
paramGrid = {
    'n_estimators': [100, 200, 300, 400, 500, 700, 1000],
    'min_samples_split': [2, 5, 10, 20],
    'max_depth': [None, 10, 20, 30, 50]
    }

gridSearch = GridSearchCV(estimator=model0, param_grid=paramGrid, cv=5, n_jobs=-1, scoring='accuracy')
gridSearch.fit(train_encoded, train0['Target'])

print(f"Optimal n_estimators: {gridSearch.best_params_['n_estimators']}")
print(f"Optimal min_samples_split: {gridSearch.best_params_['min_samples_split']}")
print(f"Optimal max_depth: {gridSearch.best_params_['max_depth']}")
print(f"Best score: {gridSearch.best_score_}")


In [ ]:
combinedValues = pd.concat([test0['Target'], predictions], axis=1)
combinedValues.columns = ['Actual', 'Predicted'] 
combinedValues.index = pd.to_datetime(combinedValues.index)

sns.set(style="whitegrid")
plt.figure(figsize=(14, 8))

plt.plot(combinedValues.index, combinedValues['Actual'], label='Actual', color='#1E96FC', linewidth=2)
plt.plot(combinedValues.index, combinedValues['Predicted'], label='Predicted', color='#FFC600', linewidth=2, linestyle='--')

plt.xlabel('Date', fontsize=12, labelpad=1)
plt.ylabel('Target Value', fontsize=12, labelpad=1)
plt.title('Actual vs Predicted Values',weight='bold', fontsize=14)
plt.legend(fontsize=12, bbox_to_anchor=(1,0.5), borderpad=.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
max_depth = getMaxDepth(tree)

title = f'First (Indiviudal) Decision Tree \nMax Depth: {max_depth}, Estimators: {model.n_estimators}, Min Samples Split: {model.min_samples_split}, Random State: {model.random_state}'

sns.set_context("notebook", font_scale=1.5)  
plt.figure(figsize=(20, 10))  
plot_tree(model0.estimators_[0], 
          filled=True, 
          feature_names=predictors, 
          class_names=model.classes_.astype(str), 
          rounded=True)
plt.title(title, fontsize=10, weight='bold', pad=5)
plt.tight_layout()
plt.show()
plt.savefig('First Decision Tree for SONY.png', dpi=300)

In [99]:
features = []
filenames = []
modelPredict =[]

checkpointFile = 'CHECKPOINTS/checkpointModel.pkl'

def saveProgress(features, filenames, modelPredict, i, totalLength):
    percentage = (i / totalLength) * 100
    with open(checkpointFile, 'wb') as f:
        pickle.dump({'features': features, 'filenames': filenames, 'modelPredict': modelPredict}, f)
    print(f"Progress saved: {percentage:.3f}%")

def loadProgress():
    if os.path.exists(checkpointFile):
        with open(checkpointFile, 'rb') as f:
            checkpoint = pickle.load(f)
        return checkpoint['features'], checkpoint['filenames'], checkpoint['modelPredict']
    else:
        return [], [], []

features, filenames, modelPredict = loadProgress()


In [ ]:
model = RandomForestClassifier(n_estimators=300, min_samples_split=7, max_depth = 10, random_state=31)

def predict(train, test, predictors, model):
    model.fit(train[predictors], train['Target'])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name='Predictions')
    combinedVals = pd.concat([test['Target'], preds], axis=1)
    return combinedVals

def backtest(data, model, predictors, saveInterval):
    allPredictions = []  
    totalLength = len(data) - 60
    startIndex = len(filenames)
    
    for i in tqdm(range(startIndex, len(data) - 60), desc="Backtesting Progress", unit="iteration"): 
        train = data.iloc[:i+60] 
        test = data.iloc[i+60:i+61]  
        
        predictions = predict(train, test, predictors, model)
        if predictions is not None and not predictions.empty:
            allPredictions.append(predictions)  # Append valid predictions

            features.append(predictions['Target'].values)
            filenames.append(test.index[0])
            modelPredict.append(predictions['Predictions'].values)
            if (i + 1 ) % saveInterval == 0:
                saveProgress(features, filenames, modelPredict, i, totalLength)
        else:
            print(f"Skipping prediction for index {i}, no valid predictions.")
    
    if not allPredictions:
        print("No predictions generated. Returning empty DataFrame.")
        return pd.DataFrame()  
    
    return pd.concat(allPredictions)

predictions = backtest(SONY_filtered, model, predictors, saveInterval=25)

In [ ]:
combinedValues = pd.concat([test['Target'], predictions], axis=1)
combinedValues.columns = ['Actual', 'Predicted'] 
combinedValues.index = pd.to_datetime(combinedValues.index)

sns.set(style="whitegrid")
plt.figure(figsize=(14, 8))

plt.plot(combinedValues.index, combinedValues['Actual'], label='Actual', color='#F3752B', linewidth=2)
plt.plot(combinedValues.index, combinedValues['Predicted'], label='Predicted', color='#ADD9F4', linewidth=2, linestyle='--')

plt.xlabel('Date', fontsize=12, labelpad=1)
plt.ylabel('Target Value', fontsize=12, labelpad=1)
plt.title('Actual vs Predicted Values',weight='bold', fontsize=14)
plt.legend(fontsize=12, bbox_to_anchor=(1,0.5), borderpad=.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
predictions['Predictions'].value_counts() #count number of predictions (0 = down, 1 = up)

In [ ]:
SONYPrecisionScore = precision_score(predictions["Target"], predictions["Predictions"])
SONYBenchmark = predictions['Target'].value_counts() / predictions.shape[0] #benchmark for precision score comparision 
print(f'SONY Precision Score = {SONYPrecisionScore}')
print(f'SONY Benchmark Data: \n{SONYBenchmark}')

In [ ]:
SONY_filtered['Target'] = SONY_filtered['Target'].astype('float64')
SONY_filtered['Close'] = pd.to_numeric(SONY_filtered['Close'], errors='coerce')
SONY_filtered['Target'] = pd.to_numeric(SONY_filtered['Target'], errors='coerce')
SONY_filtered.dropna(subset=['Close', 'Target'], inplace=True)
SONY_filtered.fillna(0, inplace=True)
SONY_filtered = SONY_filtered.drop('Close_Ratio_2', axis=1)

print(SONY_filtered.dtypes)

In [60]:
SONY_filtered1 = SONY_filtered.select_dtypes(include=['float64', 'int64'])
horizons = [1,3,12,48] #for rolling means (, 3months, year)

newPredictors = []
for horizon in horizons:
    rollingAverage = SONY_filtered1.rolling(horizon).mean()
    
    ratioColumn = f'Close_Ratio_{horizon}'
    SONY_filtered1[ratioColumn] = SONY_filtered1['Close'] / rollingAverage['Close']
    
    trendColumn = f'Trend_{horizon}'
    SONY_filtered1[trendColumn] = SONY_filtered1.shift(1).rolling(horizon).sum()['Target']
    
    newPredictors += [ratioColumn, trendColumn]

In [ ]:
SONY_filtered1.fillna(0, inplace=True)
SONY_filtered1

In [ ]:
features1 = []
filenames1 = []
modelPredict1 = []

checkpointFile = 'CHECKPOINTS/checkpointModelNew.pkl'

def saveProgress1(features1, filenames1, modelPredict1, i, totalLength1):
    percentage = (i / totalLength1) * 100
    with open(checkpointFile, 'wb') as f:
        pickle.dump({'features': features1, 'filenames': filenames1, 'modelPredict': modelPredict1}, f)
    print(f"Progress saved: {percentage:.3f}%")

def loadProgress1():
    if os.path.exists(checkpointFile):
        with open(checkpointFile, 'rb') as f:
            checkpoint = pickle.load(f)
        return checkpoint['features'], checkpoint['filenames'], checkpoint['modelPredict']
    else:
        return [], [], []

features1, filenames1, modelPredict1 = loadProgress1()

modelNew = RandomForestClassifier(n_estimators=250, min_samples_split=7, max_depth=10, random_state=31)

def predict1(train, test, predictors, model):
    model.fit(train[predictors], train['Target'])  
    preds = model.predict_proba(test[predictors])[:, 1] 
    preds[preds >= 0.6] = 1  #threshold for prediction
    preds[preds < 0.6] = 0
    preds = pd.Series(preds, index=test.index, name='Predictions')
    
    combinedVals = pd.concat([test['Target'], preds], axis=1)
    return combinedVals

def backtest1(data, predict1, predictors, modelNew, saveInterval):
    allPredictions = []  #store combined predictions
    totalLength1 = len(data) - 60
    startIndex = len(filenames1)  
    
    for i in tqdm(range(startIndex, len(data) - 60), desc="Backtesting Progress", unit="iteration"): 
        train = data.iloc[:i+60] 
        test = data.iloc[i+60:i+61] 
        
        predictions = predict1(train, test, predictors, modelNew)
        if predictions is not None and not predictions.empty:
            allPredictions.append(predictions) 
            
            features1.append(predictions['Target'].values)
            filenames1.append(test.index[0])
            modelPredict1.append(predictions['Predictions'].values)
            
            if (i + 1) % saveInterval == 0:
                saveProgress1(features, filenames1, modelPredict1, i, totalLength1)
        else:
            print(f"Skipping prediction for index {i}, no valid predictions.")
    
    if not allPredictions:
        print("No predictions generated. Returning empty DataFrame.")
        return pd.DataFrame()  
    
    return pd.concat(allPredictions)

newPredictions = backtest1(SONY_filtered1, predict1, newPredictors, modelNew, saveInterval=25)

In [ ]:
combinedValues = pd.concat([test['Target'], newPredictions], axis=1)
combinedValues.columns = ['Actual', 'Predicted'] 
combinedValues.index = pd.to_datetime(combinedValues.index)

sns.set(style="whitegrid")
plt.figure(figsize=(14, 8))

plt.plot(combinedValues.index, combinedValues['Actual'], label='Actual', color='#FAA6FF', linewidth=2)
plt.plot(combinedValues.index, combinedValues['Predicted'], label='Predicted', color='#66462C', linewidth=2, linestyle='--')

plt.xlabel('Date', fontsize=12, labelpad=1)
plt.ylabel('Target Value', fontsize=12, labelpad=1)
plt.title('Actual vs Predicted Values',weight='bold', fontsize=14)
plt.legend(fontsize=12, bbox_to_anchor=(1,0.5), borderpad=.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
newPredictions['Predictions'].value_counts()

In [ ]:
SONYPrecisionScoreNew = precision_score(newPredictions["Target"], newPredictions["Predictions"])
SONYBenchmarkNew = newPredictions['Target'].value_counts() / newPredictions.shape[0] #benchmark for precision score comparision 
print(f'SONY Precision Score = {SONYPrecisionScoreNew}')
print(f'SONY Benchmark Data: \n{SONYBenchmarkNew}')